In [6]:
# Requires: pip install cvxpy numpy scipy
import cvxpy as cp
import random
import numpy as np

def idx(i, k):     # i in 0..n-1, k in {1,2,3}
    return 3*i + (k-1)

def build_and_solve_sdp(n, edges, alpha=None, beta=None, gamma=None, solver='SCS'):
    # edges: list of (i,j,w)
    # alpha/beta/gamma: either None or dict mapping (i,j)->value in [-1,1] (default 1)
    if alpha is None: alpha = { (i,j):1 for (i,j,_) in edges }
    if beta  is None: beta  = { (i,j):1 for (i,j,_) in edges }
    if gamma is None: gamma = { (i,j):1 for (i,j,_) in edges }

    m = 3 * n
    M = cp.Variable((m,m), PSD=True)

    print(M)

    constraints = []
    # symmetry (CVXPY PSD var is symmetric numerically; enforce if you want)
    constraints += [M == M.T]

    # CONSTRAINT 1: diagonal = 1
    for i in range(n):
        for k in (1,2,3):
            constraints.append(M[idx(i,k), idx(i,k)] == 1)

    # CONSTRAINT 2: local anti-commutation: M(ik,il) = - M(il,ik) for k!=l
    for i in range(n):
        for k in (1,2,3):
            for l in (1,2,3):
                if k >= l:
                    continue
                constraints.append(M[idx(i,k), idx(i,l)] + M[idx(i,l), idx(i,k)] == 0)

    # objective
    obj_terms = []
    for (i,j,w) in edges:
        a = alpha.get((i,j), alpha.get((j,i), 1.0))
        b = beta.get((i,j),  beta.get((j,i),  1.0))
        c = gamma.get((i,j), gamma.get((j,i), 1.0))
        term = w * (1
                    - a * M[idx(i,1), idx(j,1)]
                    - b * M[idx(i,2), idx(j,2)]
                    - c * M[idx(i,3), idx(j,3)])
        obj_terms.append(term)
    objective = cp.Maximize(cp.sum(obj_terms))

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS, verbose=True)   # or solver='MOSEK' if available

    return prob.value, M.value  # SDP upper bound and numeric matrix

# Example: triangle graph with unit weights
n = 20
p = 0.2   # edge probability (sparsity)
edges = []

for i in range(n):
    for j in range(i+1, n):
        if random.random() < p:
            edges.append((i, j, 1.0))

print(f"{len(edges)} edges:", edges[:10], "...")
val, Mval = build_and_solve_sdp(n, edges, solver='SCS')
print("SDP bound:", val)

(CVXPY) Nov 13 07:59:42 PM: Your problem has 3600 variables, 3720 constraints, and 0 parameters.
(CVXPY) Nov 13 07:59:42 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Nov 13 07:59:42 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Nov 13 07:59:42 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Nov 13 07:59:42 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Nov 13 07:59:42 PM: Compiling problem (target solver=SCS).
(CVXPY) Nov 13 07:59:42 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Nov 13 07:59:42 PM: Applying reduction FlipObjective
(CVXPY) Nov 13 07:59:42 PM: Applying reduction Dcp2Cone
(CVXPY) Nov 13 07:59:42 PM: Applying reduction CvxAttr2Constr
(CVXPY) Nov 13 07:59:42 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Nov 13 07:59:42 PM: Applying 

32 edges: [(0, 7, 1.0), (0, 10, 1.0), (0, 11, 1.0), (0, 17, 1.0), (1, 3, 1.0), (1, 8, 1.0), (1, 15, 1.0), (1, 16, 1.0), (2, 8, 1.0), (2, 10, 1.0)] ...
var5207
                                     CVXPY                                     
                                     v1.7.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.9 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
---------------------------------------

In [7]:
import itertools

def max_cut_brute_force(n, edges):
    best_cut_val = -float('inf')
    best_partition = None

    # all possible 0/1 assignments of n nodes
    for bits in itertools.product([0,1], repeat=n):
        cut_val = 0
        for i,j,w in edges:
            if bits[i] != bits[j]:  # edge crosses the cut
                cut_val += w
        if cut_val > best_cut_val:
            best_cut_val = cut_val
            best_partition = bits

    return best_cut_val, best_partition

# example
val, partition = max_cut_brute_force(n, edges)
print("Brute-force Max-Cut value:", val)

Brute-force Max-Cut value: 26.0
